In [1]:
import numpy as np



words = [
    "low",
    "low",
    "lowest",
    "newer",
    "newest",
    "wide",
    "wider"
]


characters = sorted(list(set("".join(words))))

stoi = {}
itos = {}

for i, ch in enumerate(characters):
    stoi[ch] = i
    itos[i] = ch

vocab_size = len(stoi)

print("Vocabulary")
print(stoi)



token_ids = []

for word in words:

    ids = []

    for ch in word:
        ids.append(stoi[ch])

    token_ids.extend(ids)

print("\nToken IDs")
print(token_ids)


context = 4

X = []
Y = []

for i in range(len(token_ids)-context):

    X.append(token_ids[i:i+context])

    Y.append(token_ids[i+1:i+context+1])

X = np.array(X)
Y = np.array(Y)

print("\nInput Shape :",X.shape)
print("Target Shape:",Y.shape)

print("\nSample Input")
print(X[0])

print("\nSample Target")
print(Y[0])



embedding_dim = 16

embedding = np.random.randn(vocab_size,embedding_dim)

embedded = embedding[X]

print("\nEmbedding Shape")
print(embedded.shape)

Vocabulary
{'d': 0, 'e': 1, 'i': 2, 'l': 3, 'n': 4, 'o': 5, 'r': 6, 's': 7, 't': 8, 'w': 9}

Token IDs
[3, 5, 9, 3, 5, 9, 3, 5, 9, 1, 7, 8, 4, 1, 9, 1, 6, 4, 1, 9, 1, 7, 8, 9, 2, 0, 1, 9, 2, 0, 1, 6]

Input Shape : (28, 4)
Target Shape: (28, 4)

Sample Input
[3 5 9 3]

Sample Target
[5 9 3 5]

Embedding Shape
(28, 4, 16)


In [2]:


seq_len = context

position_encoding = np.zeros((seq_len, embedding_dim))

for pos in range(seq_len):
    for i in range(embedding_dim):
        angle = pos / np.power(10000, (2 * (i // 2)) / embedding_dim)

        if i % 2 == 0:
            position_encoding[pos, i] = np.sin(angle)
        else:
            position_encoding[pos, i] = np.cos(angle)

embedded = embedded + position_encoding

print("After Positional Encoding :", embedded.shape)



Wq = np.random.randn(embedding_dim, embedding_dim) * 0.01
Wk = np.random.randn(embedding_dim, embedding_dim) * 0.01
Wv = np.random.randn(embedding_dim, embedding_dim) * 0.01

Q = embedded @ Wq
K = embedded @ Wk
V = embedded @ Wv

print("Q Shape :", Q.shape)
print("K Shape :", K.shape)
print("V Shape :", V.shape)



scores = np.matmul(Q, np.transpose(K, (0,2,1)))

scores = scores / np.sqrt(embedding_dim)



mask = np.triu(np.ones((seq_len, seq_len)), k=1)

scores = np.where(mask == 1, -1e9, scores)



exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))

attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

print("Attention Weights Shape :", attention_weights.shape)



attention_output = np.matmul(attention_weights, V)

print("Attention Output Shape :", attention_output.shape)

After Positional Encoding : (28, 4, 16)
Q Shape : (28, 4, 16)
K Shape : (28, 4, 16)
V Shape : (28, 4, 16)
Attention Weights Shape : (28, 4, 4)
Attention Output Shape : (28, 4, 16)


In [3]:


W_out = np.random.randn(embedding_dim, vocab_size) * 0.01
b_out = np.zeros((1, vocab_size))


logits = attention_output @ W_out + b_out

print("Logits Shape :", logits.shape)



exp_logits = np.exp(logits - np.max(logits, axis=2, keepdims=True))

probabilities = exp_logits / np.sum(exp_logits, axis=2, keepdims=True)

print("Probability Shape :", probabilities.shape)


loss = 0

for i in range(Y.shape[0]):
    for j in range(Y.shape[1]):
        target = Y[i, j]
        loss += -np.log(probabilities[i, j, target] + 1e-10)

loss = loss / (Y.shape[0] * Y.shape[1])

print("\nCross Entropy Loss :", loss)


gradient = probabilities.copy()

for i in range(Y.shape[0]):
    for j in range(Y.shape[1]):
        gradient[i, j, Y[i, j]] -= 1

gradient = gradient / (Y.shape[0] * Y.shape[1])

learning_rate = 0.01

dW = np.matmul(
    attention_output.reshape(-1, embedding_dim).T,
    gradient.reshape(-1, vocab_size)
)

db = np.sum(gradient.reshape(-1, vocab_size), axis=0, keepdims=True)

W_out = W_out - learning_rate * dW
b_out = b_out - learning_rate * db

print("\nWeights Updated Successfully")



print("\nGenerated Tokens")

sample = X[0]

generated = list(sample)

for step in range(10):

    current = np.array(generated[-context:])

    emb = embedding[current]

    emb = emb + position_encoding

    Q = emb @ Wq
    K = emb @ Wk
    V = emb @ Wv

    score = Q @ K.T
    score = score / np.sqrt(embedding_dim)

    temp_mask = np.triu(np.ones((context, context)), k=1)

    score = np.where(temp_mask == 1, -1e9, score)

    ex = np.exp(score - np.max(score, axis=1, keepdims=True))
    att = ex / np.sum(ex, axis=1, keepdims=True)

    out = att @ V

    logit = out[-1] @ W_out + b_out

    prob = np.exp(logit - np.max(logit))
    prob = prob / np.sum(prob)

    next_token = np.argmax(prob)

    generated.append(next_token)

print(generated)

print("\nGenerated Text")

for token in generated:
    print(itos[token], end="")

Logits Shape : (28, 4, 10)
Probability Shape : (28, 4, 10)

Cross Entropy Loss : 2.3026366766822806

Weights Updated Successfully

Generated Tokens
[np.int32(3), np.int32(5), np.int32(9), np.int32(3), np.int32(3), np.int32(3), np.int32(3), np.int32(3), np.int32(3), np.int32(3), np.int32(3), np.int32(3), np.int32(3), np.int32(3)]

Generated Text
lowlllllllllll